This example fetches all pages for each username and UTC date. A search can span up to 30 days after its start date. Install `tqdm` and `pandas` in the notebook environment to display progress and build a table.

In [1]:
from datetime import datetime, timedelta


def generate_date_list(start_date: str, end_date: str) -> list[str]:
    """
    Generate a list of dates between two dates, inclusive.

    Args:
        start_date (str): The start date as a string in "YYYYMMDD" format.
        end_date (str): The end date as a string in "YYYYMMDD" format.

    Returns:
        list of str: A list containing all dates from the start date to the end date,
                     inclusive, each formatted as "YYYYMMDD".

    Raises:
        ValueError: If the input date strings are not in the correct format or if
                    the start date occurs after the end date.

    Example:
        >>> generate_date_list("20240811", "20240815")
        ['20240811', '20240812', '20240813', '20240814', '20240815']
    """
    date_format = "%Y%m%d"
    first = datetime.strptime(start_date, date_format)
    last = datetime.strptime(end_date, date_format)
    if last < first:
        raise ValueError("end_date must be on or after start_date")
    delta = last - first
    return [
        (first + timedelta(days=i)).strftime(date_format)
        for i in range(delta.days + 1)
    ]


start = "20240816"
end = "20241003"

dates = generate_date_list(start, end)


We recommend you store your client key and secret in the environment variables and call them here.

In [ ]:
from os import environ
from TikTok.Query import Query
from TikTok.Auth import OAuth2
from TikTok.ValidationModels.OAuth2 import RequestHeadersModel, TokenRequestBodyModel

credentials = TokenRequestBodyModel(
    client_key=environ["CLIENT_KEY"],
    client_secret=environ["CLIENT_SECRET"],
)


The SDK reports HTTP and API errors with `QueryException`. The example retries connection failures, HTTP 429, and server errors. It reports a failed page when the retry limit is reached.

In [ ]:
import asyncio
from itertools import batched, product

from tqdm import tqdm
from TikTok.Exceptions.Query import QueryException
from TikTok.ValidationModels.Video import (
    VideoQueryRequestBuilder,
    VideoQueryOperation,
    VideoQueryFieldName,
    VideoQueryFields,
    VideoDataModel,
)

VIDEO_FIELDS = (
    VideoQueryFields.id,
    VideoQueryFields.video_description,
    VideoQueryFields.create_time,
    VideoQueryFields.region_code,
    VideoQueryFields.share_count,
    VideoQueryFields.view_count,
    VideoQueryFields.like_count,
    VideoQueryFields.comment_count,
    VideoQueryFields.music_id,
    VideoQueryFields.hashtag_names,
    VideoQueryFields.username,
    VideoQueryFields.effect_ids,
    VideoQueryFields.playlist_id,
    VideoQueryFields.favorites_count,
)


async def fetch_videos_for_date(
    query: Query, username: str, day: str, max_retries: int
) -> list[VideoDataModel]:
    """Return all pages for one username and one UTC day in YYYYMMDD format.

    Retry connection failures, HTTP 429, and server errors up to max_retries
    times per page. Raise QueryException if the request fails. Raise ValueError
    if the search or the next page details are invalid.
    """
    request = (
        VideoQueryRequestBuilder()
        .start_date(day)
        .end_date(day)
        .max_count(100)
        .and_(VideoQueryOperation.EQ, VideoQueryFieldName.username, [username])
        .build()
    )
    videos: list[VideoDataModel] = []
    retries = 0

    while True:
        try:
            response = await query.video.search(request=request, fields=VIDEO_FIELDS)
        except QueryException as error:
            status = error.status_code
            if retries >= max_retries or (
                status is not None and status != 429 and status < 500
            ):
                raise
            retries += 1
            await asyncio.sleep(2**retries)
            continue

        retries = 0
        page = response.data
        videos.extend(page.videos)
        if not page.has_more:
            return videos
        if page.cursor is None or not page.search_id:
            raise ValueError("TikTok omitted the cursor or ID for the next page.")
        request = request.model_copy(
            update={"cursor": page.cursor, "search_id": page.search_id}
        )


Set `concurrency` to the maximum number of username and date pairs to search at the same time. The default is one. Each search reads its pages in order.

In [ ]:
async def fetch_videos_for_usernames(
    query: Query,
    usernames: list[str],
    date_list: list[str],
    max_retries: int = 3,
    *,
    concurrency: int = 1,
) -> list[VideoDataModel]:
    """Return videos in username, date, and page order.

    Supply dates in YYYYMMDD format. Set max_retries for each page and use a
    positive concurrency limit. An empty input gives an empty list. If a batch
    has errors, wait for the batch and raise the first error in input order.
    """
    videos: list[VideoDataModel] = []
    batches = batched(product(usernames, date_list), concurrency)
    with tqdm(total=len(usernames) * len(date_list), unit="search") as progress:
        for batch in batches:
            tasks = [
                asyncio.create_task(
                    fetch_videos_for_date(query, username, day, max_retries)
                )
                for username, day in batch
            ]
            await asyncio.gather(*tasks, return_exceptions=True)
            videos.extend((*task.result() for task in tasks))
            progress.update(len(batch))
    return videos


In [ ]:
profiles = ["tiktok"]
auth = OAuth2(headers=RequestHeadersModel(), body=credentials)
query = Query(auth)
async with auth.client, query.client:
    videos = await fetch_videos_for_usernames(query, profiles, dates)


Convert the selected video fields to a table. Missing values represent fields that TikTok did not return.

In [ ]:
import pandas as pd

df = pd.DataFrame.from_records(
    [video.model_dump(by_alias=True, exclude_none=True) for video in videos]
)
